# Worksheet 1 — Setup: The Tools Behind an Agent That Learns From Pixels

**Your project:** *teaching a neural network to play Atari Pong using nothing but the
screen.*

This is the worksheet version of `01_setup.ipynb`. Same material, with the sections
numbered, the outputs explained, and exercises at the end.

Read the markdown, run each code cell, and **read its output before moving on**. Several
cells exist only so that you look at a number and work out why it had to be that number.

## 1. What you will learn

1. What the agent is and is not told — and why that makes Pong hard.
2. How to set up the environment, and what `uv sync --locked` actually guarantees.
3. Which of the three libraries owns which job, so you know where to look when
   something breaks.
4. How to drive a real Atari emulator from Python: `reset()`, `step()`, and what comes
   back.
5. What "sparse reward" means when you measure it rather than read about it.

You will not train anything here. Worksheet 2 builds the network; Worksheet 3 makes it
learn.

## Running this notebook

**Locally**: the environment comes from the repo. `uv sync --locked`, then
`uv run jupyter lab`. The cell below does nothing.

**On Colab**: the kernel exists before the project does, so the cell below clones the
repo and installs what Colab is missing. Run it first.

It installs `gymnasium` and `ale-py` only. Colab already provides a working CUDA build
of torch, and replacing it with the exact locked one means downloading the entire CUDA
stack for no benefit to anything taught here. The README describes the stricter route
if you ever need bit-identical versions.


In [1]:
# Colab bootstrap. Does nothing when you run this notebook locally.
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # The -d test makes re-running this cell harmless.
    ![ -d /content/pong-pytorch ] || git clone -q https://github.com/piyushahuja/pong-pytorch.git /content/pong-pytorch
    %cd /content/pong-pytorch
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    !~/.local/bin/uv pip install -q --system --python {sys.executable} gymnasium ale-py
    !~/.local/bin/uv pip install -q --system --python {sys.executable} -e . --no-deps

print("python:", sys.version.split()[0])
if IN_COLAB:
    print("working directory:", os.getcwd())


python: 3.12.14


In [2]:
# Confirm the environment is what you think it is.
import torch
import gymnasium
import ale_py

print("torch:     ", torch.__version__)
print("gymnasium: ", gymnasium.__version__)
print("ale-py:    ", ale_py.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch:      2.14.0
gymnasium:  1.3.0
ale-py:     0.12.1
CUDA available: False


Nothing visible happened, which is the point: importing ALE does not start a game. It
registers what is available.

Worth holding onto: ALE is a pure function of state and action. Hand it a controller
input, it advances the emulated console and hands back pixels and a reward. It would
behave identically if a human, or a random number generator, were choosing the actions —
it has no idea a neural network exists.

Read those four lines before going on.

`torch` and `gymnasium` should be present, because `uv sync --locked` installed the exact
versions recorded in `uv.lock`. That is the guarantee: not "a recent torch", but *this*
torch, the one every other person working on this repo has.

`CUDA available` will say `False` on a laptop and `True` on Colab with a GPU runtime.
Either is fine. This project trains on CPU by default, and for a good reason you will see
in Worksheet 2: the network is small and processes one frame at a time, so the overhead of
talking to a GPU costs more than the GPU saves.

## 2. Project setup

There is nothing to create: the repo *is* the project. From a clone,

```bash
uv sync --locked
```

provisions Python, installs the exact versions in `uv.lock`, and installs the `pong`
package itself in editable mode — which is what makes `import pong` work in these
notebooks without any `sys.path` juggling. Then use the project `.venv` as your
notebook kernel.

`--locked` fails rather than quietly re-resolving if `pyproject.toml` and `uv.lock` have
drifted, so everyone gets the same environment. Use it every time; there is no reason to
run plain `uv sync` here.

Do not run `uv init` in a clone — it would overwrite `pyproject.toml`. That command is
for starting a project from nothing, which is not what you are doing.

To add a dependency later, `uv add <package>`. It changes two files, `pyproject.toml`
and `uv.lock`; commit both together, or `uv sync --locked` breaks for everyone else.


## 3. Same policy API, different environments

A PyTorch policy talks to the world through Gymnasium. Under that API you can plug in MuJoCo (robots) or ALE (Atari Pong). The learning loop shape stays the same.

![PyTorch policy stacks for MuJoCo vs ALE Pong](../assets/stack-policy-envs.svg)

The point of that figure is worth stating plainly, because it is why any of this
transfers: **Gymnasium's API is not Pong-specific.** Anything implementing `reset()` and
`step()` can be swapped in underneath the same agent — a robot arm in MuJoCo, a different
Atari game, a simulator you write yourself.

What changes between them is the observation you get and how you turn it into network
input. The learning loop keeps its shape.

## 4. ALE (`ale-py`)

ALE = Arcade Learning Environment. The `ale-py` package lets Gymnasium run Atari 2600 games like Pong.

It is an emulator-based research environment: it runs the actual game ROM, keeps internal state, accepts controller actions, and returns pixels and rewards.

ALE does not know about neural networks. It only simulates Atari:

$$
(\text{game state},\, \text{action})
\rightarrow
(\text{new state},\, \text{pixels},\, \text{reward})
$$

In [3]:
import ale_py

## 5. Gymnasium

Gymnasium is the RL API that connects an agent to an environment. The two calls you will use constantly:

```python
obs, info = env.reset()
obs, reward, terminated, truncated, info = env.step(action)
```

In [4]:
import gymnasium as gym

### Seeing it actually run

The cell below is not in the original notebook. It is here because reading about
`reset()` and `step()` is much less useful than watching what they return.

The repo installs itself as a package, so its own settings are importable rather than
copied. `pong.make_env()` builds the environment with exactly the frameskip and
action-repeat values training used — which matters, because a policy trained under one
set of those and evaluated under another is being tested on a different game.


In [5]:
import pong        # the project's own code, installed by `uv sync --locked`

print("frameskip:", pong.FRAMESKIP, "| action-repeat chance:", pong.STICKY)


frameskip: (2, 5) | action-repeat chance: 0.25


In [6]:
env = pong.make_env()          # the project's settings, so this matches training
observation, info = env.reset(seed=0)

print("observation shape:", observation.shape)
print("observation dtype:", observation.dtype)
print("numbers per frame:", observation.size)
print("\nactions available:", env.unwrapped.get_action_meanings())

observation shape: (210, 160, 3)
observation dtype: uint8
numbers per frame: 100800

actions available: ['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']


A.L.E: Arcade Learning Environment (version 0.12.1+8a8fafb)
[Powered by Stella]


Three separate facts are in that output.

**`(210, 160, 3)` and 100,800.** 210 rows, 160 columns, 3 colour channels. `uint8` means
each is a whole number from 0 to 255. That is everything the agent gets, every frame.

**The action list.** Six entries, and `RIGHT` and `LEFT` sit at positions 2 and 3. In Pong
those are the two that move your paddle, so the agent's entire decision each frame is:
action 2, or action 3.

**Nothing in there is about Pong.** No ball position, no velocity, no paddle coordinates,
not even a hint that some pixels matter and others are decoration. The agent has to
discover all of that from the 100,800 numbers.

In [7]:
# How often does the agent actually get told anything?
observation, info = env.reset(seed=0)

first_reward_at = None
for step in range(400):
    action = env.action_space.sample()            # random, on purpose
    observation, reward, terminated, truncated, info = env.step(action)
    if reward != 0 and first_reward_at is None:
        first_reward_at = (step, reward)
    if terminated or truncated:
        break

print("steps taken:", step + 1)
print("first nonzero reward:", first_reward_at)
print("episode finished?", terminated or truncated)

steps taken: 400
first nonzero reward: (83, -1.0)
episode finished? False


This is the sparse-reward problem, measured rather than described.

A few hundred steps, and for almost every one of them the reward was exactly `0`.
Somewhere in there a point was scored, and since the actions were random it was almost
certainly scored *against* us, so the reward was `-1`.

Now think about what that leaves you to work with. The agent made hundreds of decisions
and received one scrap of feedback, which does not say **which** decision was the bad one.
Maybe the fatal mistake was sixty frames before the ball went past; maybe by then it was
already unavoidable. Spreading one late reward back over the choices that led to it is
exactly the problem Worksheet 3 solves.

Notice the episode almost certainly did not end. A full game runs to 21 points and takes
a few thousand frames.

In [8]:
env.close()      # always release an environment you opened
print("closed")

closed


## 6. PyTorch (`torch`)

PyTorch is the deep-learning library for this project. You will use it to:

- define a policy network (`torch.nn`)
- run a forward pass from pixels to action probabilities
- compute a loss, call `.backward()`, and update weights with an optimizer

It does not simulate Pong; Gymnasium + ALE do that. Torch only owns the learnable policy and gradients.

In [9]:
import torch
import torch.nn as nn

In [10]:
# The whole idea behind a deep-learning framework, in two lines.
plain = torch.zeros(3)
learnable = torch.zeros(3, requires_grad=True)

print("plain:    ", plain, "| requires_grad:", plain.requires_grad)
print("learnable:", learnable, "| requires_grad:", learnable.requires_grad)

plain:     tensor([0., 0., 0.]) | requires_grad: False
learnable: tensor([0., 0., 0.], requires_grad=True) | requires_grad: True


Both are three zeros. The difference between them is what you are really buying when you
use PyTorch.

`requires_grad=True` makes torch remember every operation performed on that tensor, so
that later it can work backwards and tell you how changing each of those three numbers
would have changed the final result. That bookkeeping is the whole trick. Worksheet 3 uses
it to turn "we lost that point" into "these weights should move this way".

## 7. Small utilities

These show up later for math helpers and saving checkpoints.

In [11]:
import math  # e.g. sqrt
from pathlib import Path  # save checkpoints

## 8. Your exercises

Some want a written answer — double-click the markdown cell and type into it. Some want
code.

**Exercise 1.1.** Explain, in your own words, why a *single* frame of Pong is not enough
to play well even for a perfect player. What is missing from one still image, and how
might you recover it from more than one frame?

*Your written answer here. (Double-click this cell to edit it.)*

In [12]:
# Exercise 1.2 — Count the numbers.
# A raw frame is 210 x 160 x 3. The project keeps only rows 35 to 195, takes every
# second row and column, and keeps one colour channel. Work out how many numbers
# survive. Print both counts and the ratio between them.

# your code here

In [13]:
# Exercise 1.3 — Does the action-repeat noise matter?
# Build the environment twice, once with pong.make_env() and once with
# pong.make_env(sticky=0.0). For each, run random actions until the first nonzero
# reward, five times, and print the five step counts. Does removing the noise change
# anything? Run it before you answer: one run tells you nothing.

# your code here

**Exercise 1.4.** The table in section 4 says `torch` does not know what a reward is. But
in Worksheet 3 you will build a loss function out of rewards. Is the table wrong? Say
precisely what work *you* are doing that torch is not.

*Your written answer here.*

## 9. Questions to think about

No short answers, and you are not expected to resolve them now. Notice which ones make you
uncomfortable — that is usually where the interesting part is.

1. The agent receives 100,800 numbers and needs perhaps six of them: the ball's position
   and velocity, and the two paddle positions. If you already knew those six, the problem
   would be nearly trivial. So is learning from pixels a real achievement, or an elaborate
   way of recovering six numbers?

2. A reward arrives long after the action that earned it. How would *you* decide which of
   the preceding few hundred decisions deserves the credit? Try to describe a rule before
   you are shown one.

3. Sticky actions make the game noisy deliberately, which makes learning harder. Why would
   anyone want the problem to be harder?

4. ALE knows nothing about neural networks; torch knows nothing about Pong. What would go
   wrong if one library owned both jobs? Think about who would have to change when you
   swap Pong for a robot arm.

5. The repo pins a Python version and records exact library versions. What experiment could
   you run in six months that would be impossible without those two files?

## 10. What you have learnt

| You can now | Concretely |
|---|---|
| State the problem precisely | 100,800 numbers in, one of two actions out, reward almost always 0 |
| Set up a reproducible environment | `uv sync --locked`, and why `pyproject.toml` and `uv.lock` are both needed |
| Say which library owns what | ALE simulates, Gymnasium mediates, torch learns |
| Drive an Atari game from Python | `make_env`, `reset(seed=...)`, `step(action)`, `close()` |
| Read an observation | shape `(210, 160, 3)`, dtype `uint8`, actions 2 and 3 for up and down |
| Explain a design choice | why sticky actions exist, and why they are part of what a trained policy means |
| Recognise what makes this hard | the input is far too big, and the reward is far too rare |
| Name PyTorch's actual job | tracking gradients, via `requires_grad` |

**Next:** `02_neural_network_worksheet.ipynb` builds the network that turns those pixels
into a decision.